# Drone CV — Multi-Platform Detection & Safety Monitoring

**Author:** Richard Wirén, Lead Solution Architect — Ericsson

**Platforms:** DJI M2EA | Autel MAX 4T V2 xe | DJI Avata 360

**Patent:** WO2025034145A1 — Calculating Lateral Distance from UAV to Object

---

## Overview
This notebook provides GPU-accelerated training and inference for the drone detection pipeline.

- **Training:** YOLOv8s at imgsz=1280 on VisDrone + Autel campus data (A100 GPU)
- **360° Processing:** Avata 360 equirectangular → perspective tiling → person detection
- **Validation:** 1:1 rule lateral distance across all platforms

In [ ]:
# === SETUP ===
!pip install -q ultralytics sahi opencv-python-headless folium paho-mqtt
!nvidia-smi

In [ ]:
# === CLONE REPO ===
# NOTE: For Ericsson internal GitLab, use SSH key or deploy token
# For development, upload the dataset zip manually
import os
if not os.path.exists('drone-cv-parking'):
    # Option A: Clone from internal GitLab (needs auth)
    # !git clone git@gitlab.internal.ericsson.com:lmfwire/detection-with-drone.git drone-cv-parking
    
    # Option B: Upload dataset.zip to Colab and extract
    print('Upload dataset.zip or clone repo manually')
else:
    print('Repo already present')
os.chdir('drone-cv-parking')

## 1. GPU Training — YOLOv8 at imgsz=1280

Training at 1280px eliminates the need for SAHI slicing on high-res Autel images (4000×3000).

In [ ]:
from ultralytics import YOLO

# Fresh YOLOv8s from COCO pretrained
model = YOLO('yolov8s.pt')

# Train on combined dataset
results = model.train(
    data='data/autel_training/combined_dataset.yaml',
    epochs=30,
    imgsz=1280,          # High-res: no SAHI needed at inference
    batch=16,            # A100 can handle batch=16 at 1280
    device=0,            # GPU
    workers=4,
    patience=10,
    project='runs/gpu_training',
    name='visdrone_autel_1280',
    exist_ok=True,
    mosaic=1.0,
    mixup=0.1,
    cos_lr=True,
)

print(f'Best model: {results.save_dir}/weights/best.pt')
print(f'mAP50: {results.results_dict["metrics/mAP50(B)"]:.3f}')
print(f'mAP50 cars: check results.csv for per-class metrics')

## 2. DJI Avata 360° Processing

Extract perspective views from equirectangular video and detect persons omnidirectionally.

In [ ]:
import cv2
import numpy as np

def extract_perspective(equirect, fov_deg=90, yaw_deg=0, pitch_deg=0, out_size=(960, 540)):
    """Extract rectilinear perspective crop from equirectangular frame."""
    h, w = equirect.shape[:2]
    out_w, out_h = out_size
    f = out_w / (2 * np.tan(np.radians(fov_deg) / 2))

    u = np.arange(out_w, dtype=np.float64) - out_w / 2
    v = np.arange(out_h, dtype=np.float64) - out_h / 2
    u, v = np.meshgrid(u, v)

    x, y, z = u, v, np.full_like(u, f)
    norm = np.sqrt(x**2 + y**2 + z**2)
    x, y, z = x/norm, y/norm, z/norm

    cp, sp = np.cos(np.radians(pitch_deg)), np.sin(np.radians(pitch_deg))
    y, z = cp*y - sp*z, sp*y + cp*z

    cy, sy = np.cos(np.radians(yaw_deg)), np.sin(np.radians(yaw_deg))
    x, z = cy*x + sy*z, -sy*x + cy*z

    lon = np.arctan2(x, z)
    lat = np.arcsin(np.clip(y, -1, 1))

    src_x = ((lon / np.pi + 1) / 2 * w).astype(np.float32)
    src_y = ((0.5 - lat / np.pi) * h).astype(np.float32)

    return cv2.remap(equirect, src_x, src_y, cv2.INTER_LINEAR, borderMode=cv2.BORDER_WRAP)


# Process 360° video (use .LRF for dev, .OSV for production)
VIDEO_PATH = 'path/to/DJI_20260612150146_0003_D.LRF'  # or .OSV

cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS)
print(f'Video: {int(cap.get(cv2.CAP_PROP_FRAME_COUNT))} frames at {fps:.0f}fps')

# Extract 8 views from a sample frame
cap.set(cv2.CAP_PROP_POS_FRAMES, int(96 * fps))  # ~1:36 (9m altitude)
ret, frame = cap.read()

model = YOLO('yolov8s.pt')
for yaw in range(0, 360, 45):
    view = extract_perspective(frame, fov_deg=90, yaw_deg=yaw, pitch_deg=-50)
    results = model(view, conf=0.25, classes=[0], verbose=False)[0]
    if len(results.boxes) > 0:
        conf = max(float(b.conf) for b in results.boxes)
        print(f'  Yaw {yaw:3d}°: {len(results.boxes)} person(s), conf={conf:.2f}')

cap.release()

## 3. Results & Validation

Compare detection performance across all three platforms.

In [ ]:
# Summary table
print('=' * 70)
print('MULTI-PLATFORM DETECTION COMPARISON')
print('=' * 70)
print(f'{"Platform":<25} {"Detection":<20} {"Distance":<15} {"Coverage"}')
print('-' * 70)
print(f'{"DJI M2EA":<25} {"VisDrone YOLO":<20} {"GSD formula":<15} Single dir (gimbal)')
print(f'{"Autel MAX 4T V2 xe":<25} {"Onboard AI + YOLO":<20} {"LRF laser":<15} Single dir (gimbal)')
print(f'{"DJI Avata 360":<25} {"COCO YOLO":<20} {"GSD from SRT":<15} 360° omnidirectional')
print('=' * 70)